In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
from pyspark.sql import Row

import datetime
users = [
    {
        "id": 1,
        "first_name": "Corrie",
        "last_name": "Van den Oord",
        "email": "cvandenoord0@etsy.com",
        "phone_numbers": Row(mobile="+1 234 567 8901", home="+1 234 567 8911"),
        "courses": [1, 2],
        "is_customer": True,
        "amount_paid": 1000.55,
        "customer_from": datetime.date(2021, 1, 15),
        "last_updated_ts": datetime.datetime(2021, 2, 10, 1, 15, 0)
    },
    {
        "id": 2,
        "first_name": "Nikolaus",
        "last_name": "Brewitt",
        "email": "nbrewitt1@dailymail.co.uk",
        "phone_numbers":  Row(mobile="+1 234 567 8923", home="1 234 567 8934"),
        "courses": [3],
        "is_customer": True,
        "amount_paid": 900.0,
        "customer_from": datetime.date(2021, 2, 14),
        "last_updated_ts": datetime.datetime(2021, 2, 18, 3, 33, 0)
    },
    {
        "id": 3,
        "first_name": "Orelie",
        "last_name": "Penney",
        "email": "openney2@vistaprint.com",
        "phone_numbers": Row(mobile="+1 714 512 9752", home="+1 714 512 6601"),
        "courses": [2, 4],
        "is_customer": True,
        "amount_paid": 850.55,
        "customer_from": datetime.date(2021, 1, 21),
        "last_updated_ts": datetime.datetime(2021, 3, 15, 15, 16, 55)
    },
    {
        "id": 4,
        "first_name": "Ashby",
        "last_name": "Maddocks",
        "email": "amaddocks3@home.pl",
        "phone_numbers": Row(mobile=None, home=None),
        "courses": [],
        "is_customer": False,
        "amount_paid": None,
        "customer_from": None,
        "last_updated_ts": datetime.datetime(2021, 4, 10, 17, 45, 30)
    },
    {
        "id": 5,
        "first_name": "Kurt",
        "last_name": "Rome",
        "email": "krome4@shutterfly.com",
        "phone_numbers": Row(mobile="+1 817 934 7142", home=None),
        "courses": [],
        "is_customer": False,
        "amount_paid": None,
        "customer_from": None,
        "last_updated_ts": datetime.datetime(2021, 4, 2, 0, 55, 18)
    }
]



In [ ]:
import pandas as pd

df_users = spark.createDataFrame(pd.DataFrame(users))

In [ ]:
df_users.show()

In [ ]:
df_users.printSchema()

In [ ]:
df_users.select('*').show()

In [ ]:
df_users.select('id', 'first_name', 'last_name').show()

In [ ]:
df_users.select(['id', 'first_name', 'last_name']).show()

In [ ]:
# Задаем alias для работы с DF
df_users.alias('u').select('u.*').show()

In [ ]:
df_users.alias('u').select('u.id', 'u.first_name', 'u.last_name').show()

In [ ]:
from pyspark.sql.functions import col

df_users.select(col('id'), 'first_name', 'last_name').show()

In [ ]:
#Пример преобразования колонки с использованием select и функций lit, concat
#На основе данных двух колонок создаем новую 
from pyspark.sql.functions import col, concat, lit

df_users.select(
    col('id'), 
    'first_name', 
    'last_name',
    concat(col('first_name'), lit(', '), col('last_name')).alias('full_name')
).show()

In [ ]:
#Также мы можем сначала описать желаемые действия, а потом уже использовать их внутри select
#Пример с созданием колонки full_name
full_name_col = concat(col('first_name'), lit(', '), col('last_name'))
full_name_alias = full_name_col.alias('full_name')
df_users.select('id', full_name_alias).show()

In [ ]:
#Создаем TempView чтобы обращаться к нашему DF, используя SQL
df_users.createOrReplaceTempView('users')

In [ ]:
#Работаем с DF и функциями Spark через SQL
spark.sql("""
    SELECT id, first_name, last_name,
        concat(first_name, ', ', last_name) AS full_name
    FROM users
"""). \
    show()

In [ ]:
#Также мы можем использовать selectExpr для вызова и использования Spark SQL функций
df_users.selectExpr('id', 'first_name', 'last_name', "concat(first_name, ', ', last_name) AS full_name").show()

In [ ]:
#Не забываем выполнять spark.stop() для того чтобы освободить ресурсы на кластере.
spark.stop()